# Miner Tips 01: Public PPL and Packaging

This notebook shows the minimum scoring loop miners should understand before submitting to the binary or ternary competitions.

It uses toy logits by default, so it runs anywhere. Replace the toy section with your own public/dev text and local candidate model when you are ready.

## Scoring shape

```text
token ids:      [t0, t1, t2, t3, ...]
model logits:   predicts t1 from t0, t2 from t1, t3 from t2, ...
loss mask:      ignore padding and prompt positions you do not want to score
metric:         perplexity = exp(mean cross entropy)
```

In [ ]:
import math
from pathlib import Path
import hashlib
import zipfile

try:
    import torch
    import torch.nn.functional as F
except ImportError:
    torch = None
    F = None

print("torch available:", torch is not None)

In [ ]:
def shifted_cross_entropy_from_logits(logits, labels, ignore_index=-100):
    """Compute standard next-token cross entropy.

    logits shape: [batch, sequence, vocab]
    labels shape: [batch, sequence]

    We drop the final logit because there is no next token after the final position.
    We drop the first label because position 0 is predicted by no previous token.
    """
    if torch is None:
        raise RuntimeError("Install torch to run this cell.")

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    flat_logits = shift_logits.view(-1, shift_logits.size(-1))
    flat_labels = shift_labels.view(-1)
    return F.cross_entropy(flat_logits, flat_labels, ignore_index=ignore_index)


def perplexity_from_loss(loss):
    """Convert cross entropy to perplexity, with a small overflow guard."""
    value = float(loss)
    if value > 80:
        return float("inf")
    return math.exp(value)

In [ ]:
# Tiny toy example: two sequences, four positions, five-token vocabulary.
# The correct next tokens are intentionally easy so the PPL is low.
if torch is not None:
    labels = torch.tensor([
        [0, 1, 2, 3],
        [1, 2, 3, 4],
    ])

    logits = torch.zeros((2, 4, 5))
    for batch in range(labels.size(0)):
        for pos in range(labels.size(1) - 1):
            next_token = labels[batch, pos + 1]
            logits[batch, pos, next_token] = 5.0

    loss = shifted_cross_entropy_from_logits(logits, labels)
    print("toy loss:", round(float(loss), 4))
    print("toy ppl:", round(perplexity_from_loss(loss), 4))
else:
    print("Install torch to run the toy PPL example.")

## Replacing the toy section with a real public/dev run

Use the task repository's reference tokenizer/model guidance. Keep this pattern:

```python
# Pseudocode only.
texts = load_public_or_self_created_eval_texts()
tokenizer = AutoTokenizer.from_pretrained(reference_qwen_tokenizer_id)
model = AutoModelForCausalLM.from_pretrained(local_candidate_model_path)
loss = average_shifted_cross_entropy(model, tokenizer, texts)
ppl = exp(loss)
```

Do not tune against private validator data. Your local public/dev score is a smoke test and ranking proxy, not the final score.

In [ ]:
def sha256_file(path):
    """Return the SHA-256 hex digest for one file."""
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def zip_directory_deterministic(source_dir, zip_path):
    """Create a stable-ish zip for a model artifact directory.

    This avoids hidden notebook outputs, caches, and local machine files.
    It is intentionally small and boring.
    """
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)

    skip_names = {".git", "__pycache__", ".ipynb_checkpoints"}
    skip_suffixes = {".pyc", ".log"}

    files = []
    for path in source_dir.rglob("*"):
        if not path.is_file():
            continue
        if any(part in skip_names for part in path.parts):
            continue
        if path.suffix in skip_suffixes:
            continue
        files.append(path)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(files):
            arcname = path.relative_to(source_dir).as_posix()
            zf.write(path, arcname)

    return {
        "zip_path": str(zip_path),
        "artifact_size_bytes": zip_path.stat().st_size,
        "artifact_sha256": sha256_file(zip_path),
    }

print("Packaging helpers loaded. Call zip_directory_deterministic('model_dir', 'artifact.zip') when ready.")

## Submission sanity checklist

- Artifact URL is public and stable.
- SHA-256 matches the exact downloadable file.
- Byte size matches the exact downloadable file.
- Model loads without notebook state, local caches, or private files.
- Summary says whether the artifact targets the binary or ternary competition.